In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3080 Laptop GPU


# 1. Pre-process

Load data

In [2]:
import pandas as pd
import numpy as np
import os
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
import time
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

df = pd.read_csv("dataset/S025_whole_df.csv")
df.head()

,TIMESTAMP,BVP,ACC_X,ACC_Y,ACC_Z,TEMP,EDA,HR,IBI,Sleep_Stage,Obstructive_Apnea,Central_Apnea,Hypopnea,Multiple_Events
0,0.000000,140.76,-40.0,7.0,50.0,33.59,0.034544,60.0,NaN,P,NaN,NaN,NaN,NaN
1,0.015625,135.32,-40.0,7.0,50.0,33.59,0.034544,60.0,NaN,P,NaN,NaN,NaN,NaN
2,0.031250,130.88,-41.0,7.0,49.0,33.59,0.034544,60.0,NaN,P,NaN,NaN,NaN,NaN
3,0.046875,118.15,-41.0,7.0,49.0,33.59,0.034544,60.0,NaN,P,NaN,NaN,NaN,NaN
4,0.062500,92.38,-41.0,8.0,50.0,33.59,0.034544,60.0,NaN,P,NaN,NaN,NaN,NaN


Clear previous results

In [3]:
checkpoint_dir = './checkpoints/'

if os.path.exists(checkpoint_dir):
    for item in os.listdir(checkpoint_dir):
        if 'sleep_stage_test' in item:
            dir_path = os.path.join(checkpoint_dir, item)
            shutil.rmtree(dir_path)
            print(f": {dir_path}")

print("Model reseted")

files_to_delete = [
    "X_train.npy", "y_train.npy",
    "X_val.npy", "y_val.npy",
    "X_test.npy", "y_test.npy"
]

save_dir = "./dataset/sleep_data_ready/"
os.makedirs(save_dir, exist_ok=True)
print("Deleteing...")
for file in files_to_delete:
    file_path = os.path.join(save_dir, file)
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f" - Cleaned: {file}")
    else:
        print(f" - skip: {file}")

if os.path.exists('./test_results/'):
    shutil.rmtree('./test_results/')
    print("test_results deleted")

: ./checkpoints/classification_sleep_stage_test_TimesNet_sleep_ftM_sl60_ll48_pl0_dm32_nh8_el2_dl1_df64_expand2_dc4_fc1_ebtimeF_dtTrue_test_0
Model reseted
Deleteing...
 - Cleaned: X_train.npy
 - Cleaned: y_train.npy
 - Cleaned: X_val.npy
 - Cleaned: y_val.npy
 - Cleaned: X_test.npy
 - Cleaned: y_test.npy
test_results deleted


Select Feature and Target, check missing value

In [4]:
features = ['BVP', 'IBI', 'EDA', 'TEMP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'HR']
label_col = 'Sleep_Stage'

stage_counts = df['Sleep_Stage'].value_counts(dropna=False)
print(stage_counts)

missing_before = df[features].isna().sum().sum()
print(f" Missing Before: {missing_before} ")

Sleep_Stage
N2    858240
W     412801
P     406592
R     240000
N1     74880
Name: count, dtype: int64
 Missing Before: 2488 


Fill the missing value by taking average of the previous and the next value

In [5]:
df[features] = df[features].interpolate(method='linear', limit_direction='both')
missing_after = df[features].isna().sum().sum()
print(f"Missing After:{missing_after}")

Missing After:0


Slice Data

In [6]:
window_size = 3840
n_samples = len(df) // window_size

X_list = []
y_list = []

for i in range(n_samples):
    start_idx = i * window_size
    end_idx = start_idx + window_size
    
    window_df = df.iloc[start_idx:end_idx]
    
    mode_series = window_df[label_col].mode()
    if mode_series.empty:
        continue
    label = mode_series.iloc[0]
    
    if label in ['P','Missing']:
        continue
        
    downsampled_data = window_df[features].groupby(np.arange(len(window_df)) // 3840).mean().values    
    X_list.append(downsampled_data)
    y_list.append(label)

X = np.array(X_list)
y = np.array(y_list)

print("X shape", X.shape) 
print("y shape:", y.shape)
print("Label head", y[:5])

X shape (412, 1, 8)
y shape: (412,)
Label head ['W' 'W' 'W' 'W' 'W']


In [7]:
label_map = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'R': 4}
y_encoded = np.array([label_map[label] for label in y])

print("label to num head", y_encoded[:5])

label to num head [0 0 0 0 0]


# 2. Single patient training

Split and save data

In [8]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y_encoded, test_size=0.2, shuffle=False, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, shuffle=False, random_state=42)

print(" X_train:", X_train.shape, "y_train:", y_train.shape)
print(" X_val:", X_val.shape, "y_val:", y_val.shape)
print(" X_test:", X_test.shape, "y_test:", y_test.shape)

np.save(os.path.join(save_dir, "X_train.npy"), X_train)
np.save(os.path.join(save_dir, "y_train.npy"), y_train)
np.save(os.path.join(save_dir, "X_val.npy"), X_val)
np.save(os.path.join(save_dir, "y_val.npy"), y_val)
np.save(os.path.join(save_dir, "X_test.npy"), X_test)
np.save(os.path.join(save_dir, "y_test.npy"), y_test)

print(f"Data saved: {save_dir}")

 X_train: (287, 1, 8) y_train: (287,)
 X_val: (42, 1, 8) y_val: (42,)
 X_test: (83, 1, 8) y_test: (83,)
Data saved: ./dataset/sleep_data_ready/


Train

In [9]:
!python -X utf8 run.py --task_name classification --is_training 1 --model_id sleep_stage_test --model TimesNet --data sleep --root_path ./dataset/sleep_data_ready/ --seq_len 3840 --enc_in 8 --c_out 5 --batch_size 16 --d_model 32 --d_ff 64 --num_workers 0 --dropout 0.5 --learning_rate 0.00001 --train_epochs 20 --patience 5

Using GPU
Args in experiment:
Basic Config
  Task Name:          classification      Is Training:        1                   
  Model ID:           sleep_stage_test    Model:              TimesNet            

Data Loader
  Data:               sleep               Root Path:          ./dataset/sleep_data_ready/
  Data Path:          ETTh1.csv           Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Model Parameters
  Top k:              5                   Num Kernels:        6                   
  Enc In:             8                   Dec In:             7                   
  C Out:              5                   d model:            32                  
  n heads:            8                   e layers:           2                   
  d layers:           1                   d FF:               64                  
  Moving Avg:         25                  Fact

Traceback (most recent call last):
  File "run.py", line 235, in <module>
    exp.train(setting)
  File "C:\Users\15810\Time-Series-Library\exp\exp_classification.py", line 112, in train
    outputs = self.model(batch_x, padding_mask, None, None)
  File "C:\Users\15810\anaconda3\envs\timesnet\lib\site-packages\torch\nn\modules\module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "C:\Users\15810\anaconda3\envs\timesnet\lib\site-packages\torch\nn\modules\module.py", line 1562, in _call_impl
    return forward_call(*args, **kwargs)
  File "C:\Users\15810\Time-Series-Library\models\TimesNet.py", line 213, in forward
    dec_out = self.classification(x_enc, x_mark_enc)
  File "C:\Users\15810\Time-Series-Library\models\TimesNet.py", line 188, in classification
    enc_out = self.layer_norm(self.model[i](enc_out))
  File "C:\Users\15810\anaconda3\envs\timesnet\lib\site-packages\torch\nn\modules\module.py", line 1553, in _wrapped_call_impl
    return 

Graph

In [10]:
results_dir = './test_results/'
subdirs = [os.path.join(results_dir, d) for d in os.listdir(results_dir) if os.path.isdir(os.path.join(results_dir, d))]
subdirs.sort(key=os.path.getmtime)
latest_subdir = subdirs[-1]

pred_path = os.path.join(latest_subdir, 'pred.npy')
true_path = os.path.join(latest_subdir, 'true.npy')

y_pred = np.load(pred_path)
y_true = np.load(true_path)
if len(y_pred.shape) > 1 and y_pred.shape[1] > 1:
    y_pred = np.argmax(y_pred, axis=1)
labels = ['W', 'N1', 'N2', 'N3', 'R']

#Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
            
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Sleep Stage Classification Confusion Matrix')
plt.xlabel('Predicted Stage (TimesNet)')
plt.ylabel('True Stage (Doctor)')
plt.show()

# Hypnogram)
clinical_order = [0, 4, 1, 2, 3] # W, R, N1, N2, N3
y_true_clinical = [-clinical_order.index(val) for val in y_true]
y_pred_clinical = [-clinical_order.index(val) for val in y_pred]
clinical_labels = ['W', 'R', 'N1', 'N2', 'N3']


display_length = min(300, len(y_true))
y_true_display = y_true_clinical[:display_length]
y_pred_display = y_pred_clinical[:display_length]

plt.figure(figsize=(14, 5))
            
plt.step(range(display_length), y_true_display, label='True Stage (Doctor)', color='black', linewidth=2.5, where='post')
            
plt.step(range(display_length), [y - 0.1 for y in y_pred_display], label='Predicted (TimesNet)', color='red', linestyle='--', linewidth=2, where='post', alpha=0.8)

plt.yticks([-0, -1, -2, -3, -4], clinical_labels)
plt.title(f'Clinical Hypnogram over Time (First {display_length} Minutes of Test Set)')
plt.xlabel('Time (Minutes)')
plt.ylabel('Sleep Stage')
plt.legend(loc='lower left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()

FileNotFoundError: [WinError 3] 系统找不到指定的路径。: './test_results/'

# 3. Cross patient training

Seleted patient at average for validation and average set

In [ ]:
info_df = pd.read_csv('./dataset/participant_info.csv', dtype=str)
info_df['SID'] = info_df['SID'].astype(str).str.strip()

info_df = info_df.dropna(subset=['SID'])
info_df['SID'] = info_df['SID'].astype(str).str.strip()
info_df = info_df[info_df['SID'].str.lower() != 'nan']
info_df = info_df[info_df['SID'] != '']

features = ['AGE', 'BMI', 'OAHI', 'AHI', 'Mean_SaO2', 'Arousal Index']

if 'Mean_SaO2' in info_df.columns:
    info_df['Mean_SaO2'] = info_df['Mean_SaO2'].astype(str).str.replace('%', '', regex=False)


for col in features:
    info_df[col] = pd.to_numeric(info_df[col], errors='coerce')


global_means = info_df[features].mean()
global_stds = info_df[features].std()

for col in features:
    print(f" - {col}: Mean = {global_means[col]:.2f}, Std = {global_stds[col]:.2f}")


def calculate_multidimensional_distance(row):
    dist_sq = 0
    for col in features:
        if pd.isna(row[col]):
            continue
        z_score = (row[col] - global_means[col]) / global_stds[col]
        dist_sq += z_score ** 2  
    return np.sqrt(dist_sq)

info_df['Distance_to_Centroid'] = info_df.apply(calculate_multidimensional_distance, axis=1)

best_multidim_average = info_df.loc[info_df['Distance_to_Centroid'].idxmin()]

print("\n--- top 5 ---")
top5_candidates = info_df.sort_values(by='Distance_to_Centroid').head(5)
print(top5_candidates[['SID'] + features + ['Distance_to_Centroid']])

In [ ]:
files_to_delete = [
    "X_train.npy", "y_train.npy",
    "X_val.npy", "y_val.npy",
    "X_test.npy", "y_test.npy"
]

print("Deleting...")
for file in files_to_delete:
    file_path = os.path.join(save_dir, file)
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f" Cleaned: {file}")
    else:
        print(f" - Skip: {file}")

Declear train set, val set, and test sets

In [ ]:
data_folder = './dataset/' 

#train
train_sids = ['S010', 'S011', 'S028', 'S081']  

#test
val_sid = 'S054'
test_healthy_sid = 'S049'                      
test_severe_sid = 'S046'
test_avg_sid = 'S014'

def process_single_patient(sid, data_dir='./dataset/', window_size=3840):
    print(f"---running on {sid}---")
    file_path = os.path.join(data_dir, f"{sid}_whole_df.csv") 
    
    df = pd.read_csv(file_path)
    
    features_cols = ['BVP', 'IBI', 'EDA', 'TEMP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'HR']
    label_col = 'Sleep_Stage'
    
    missing_before = df[features_cols].isna().sum().sum()
    print(f"Missing before: {missing_before}")
    
    df[features_cols] = df[features_cols].interpolate(method='linear', limit_direction='both')
    
    
    missing_after = df[features_cols].isna().sum().sum()
    print(f"Missing after: {missing_after}") 
    
    scaler = StandardScaler()
    df[features_cols] = scaler.fit_transform(df[features_cols])
    
    return df


print("------test signle------")
df_healthy = process_single_patient(test_healthy_sid)
df_severe = process_single_patient(test_severe_sid)
df_avg = process_single_patient(test_avg_sid)
df_val = process_single_patient(val_sid)
print("------train set------")
train_dfs = {} 

for sid in train_sids:
    train_dfs[sid] = process_single_patient(sid)

Slice

In [ ]:
def create_windows(df, window_size=3840):
    features_cols = ['BVP', 'IBI', 'EDA', 'TEMP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'HR']
    label_col = 'Sleep_Stage'
    
    n_samples = len(df) // window_size
    X_list = []
    y_list = []
    
    for i in range(n_samples):
        start_idx = i * window_size
        end_idx = start_idx + window_size
        
        window_df = df.iloc[start_idx:end_idx]
        
        mode_series = window_df[label_col].mode()
        if mode_series.empty:
            continue
        label = mode_series.iloc[0]
        
        if str(label).strip() in ['P', 'Missing']:
            continue
            
        downsampled_data = window_df[features_cols].groupby(np.arange(len(window_df)) // 3840).mean().values
        
        X_list.append(downsampled_data)
        y_list.append(label)

    return np.array(X_list), np.array(y_list)

print("------ single slice ------")
X_val, y_val = create_windows(df_val)
print(f"val X: {X_val.shape}, y: {y_val.shape}")

X_test_healthy, y_test_healthy = create_windows(df_healthy)
print(f"healthy test X: {X_test_healthy.shape}, y: {y_test_healthy.shape}")

X_test_severe, y_test_severe = create_windows(df_severe)
print(f"severe test X: {X_test_severe.shape}, y: {y_test_severe.shape}")

X_test_avg, y_test_avg = create_windows(df_avg)
print(f"avg test X: {X_test_avg.shape}, y: {y_test_avg.shape}")

print("\n------ group slice ------")
X_train_list = []
y_train_list = []

for sid in train_sids:
    print(f"sclicing data for {sid}...")
    
    clean_df = train_dfs[sid] 
    X_temp, y_temp = create_windows(clean_df)
    
    X_train_list.append(X_temp)
    y_train_list.append(y_temp)

X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

print(f"\n X_train: {X_train.shape}, y_train: {y_train.shape}")

Encode data and start testing with average set

In [ ]:
label_map = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'R': 4}

def encode_labels(y_array, mapping):
    return np.array([mapping[label] for label in y_array])

y_train_encoded = encode_labels(y_train, label_map)
y_val_encoded = encode_labels(y_val, label_map)
y_test_healthy_encoded = encode_labels(y_test_healthy, label_map)
y_test_severe_encoded = encode_labels(y_test_severe, label_map)
y_test_avg_encoded = encode_labels(y_test_avg, label_map)

print(f"lable changed: {y_train_encoded[:5]}")

print("\n------ shape varify ------")
print(f"X_train: {X_train.shape}, y_train: {y_train_encoded.shape}")
print(f"X_val:   {X_val.shape}, y_val:   {y_val_encoded.shape}")
print(f"X_test_healthy: {X_test_healthy.shape}, y_test_healthy: {y_test_healthy_encoded.shape}")
print(f"X_test_severe:  {X_test_severe.shape}, y_test_severe:  {y_test_severe_encoded.shape}")
print(f"X_test_avg:  {X_test_avg.shape}, y_test_avg:  {y_test_avg_encoded.shape}")

#save
save_dir = "./dataset/sleep_data_ready/"
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, "X_train.npy"), X_train)
np.save(os.path.join(save_dir, "y_train.npy"), y_train_encoded)
np.save(os.path.join(save_dir, "X_val.npy"), X_val)
np.save(os.path.join(save_dir, "y_val.npy"), y_val_encoded)
np.save(os.path.join(save_dir, "X_test.npy"), X_test_avg)
np.save(os.path.join(save_dir, "y_test.npy"), y_test_avg_encoded)


print(f"\nData saved: {save_dir}")
print("ready for training and average test")

In [ ]:
def check_label_distribution(y_array, dataset_name):
    counts = Counter(y_array)

    reverse_map = {0: 'W', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'R'}
    
    print(f" --- {dataset_name}  ---")
    total = len(y_array)
    if total == 0:
        print("empty \n")
        return
        
    for num_label in range(5):
        count = counts.get(num_label, 0)
        percentage = (count / total) * 100
        print(f"  {reverse_map[num_label]:<2} ({num_label}): {count:<5} ({percentage:>5.1f}%)")
    print(f" Total: {total} \n")

print("\n Checking Status：\n")
check_label_distribution(y_train_encoded, "Train")
check_label_distribution(y_val_encoded, "Val")
check_label_distribution(y_test_avg_encoded, "Test Avg (S014)")
check_label_distribution(y_test_healthy_encoded, "Test Healthy (S049)")
check_label_distribution(y_test_severe_encoded, "Test Severe (S046)")

In [ ]:
checkpoint_dir = './checkpoints/'

if os.path.exists(checkpoint_dir):
    for item in os.listdir(checkpoint_dir):
        if 'sleep_stage_test' in item:
            dir_path = os.path.join(checkpoint_dir, item)
            shutil.rmtree(dir_path)
            print(f"Removed: {dir_path}")

print("Model Cleaned")

if os.path.exists('./test_results/'):
    shutil.rmtree('./test_results/')
    print("test_results deleted")

Train1

In [ ]:
#!python -X utf8 run.py --task_name classification --is_training 1 --model_id sleep_stage_test --model TimesNet --data sleep --root_path ./dataset/sleep_data_ready/ --seq_len 3840 --enc_in 8 --c_out 5 --batch_size 16 --d_model 16 --d_ff 32 --num_workers 0

In [ ]:
checkpoint_dir = './checkpoints/'

if os.path.exists(checkpoint_dir):
    for item in os.listdir(checkpoint_dir):
        if 'sleep_stage_test' in item:
            dir_path = os.path.join(checkpoint_dir, item)
            shutil.rmtree(dir_path)
            print(f"Removed: {dir_path}")

print("Model Cleaned")

if os.path.exists('./test_results/'):
    shutil.rmtree('./test_results/')
    print("test_results deleted")

Train2

In [ ]:
#!python -X utf8 run.py --task_name classification --is_training 1 --model_id sleep_stage_test --model TimesNet --data sleep --root_path ./dataset/sleep_data_ready/ --seq_len 3840 --enc_in 8 --c_out 5 --batch_size 16 --d_model 16 --d_ff 32 --num_workers 0 --dropout 0.4 --learning_rate 0.00005 --train_epochs 20 --patience 5

In [ ]:
checkpoint_dir = './checkpoints/'

if os.path.exists(checkpoint_dir):
    for item in os.listdir(checkpoint_dir):
        if 'sleep_stage_test' in item:
            dir_path = os.path.join(checkpoint_dir, item)
            shutil.rmtree(dir_path)
            print(f"Removed: {dir_path}")

print("Model Cleaned")

if os.path.exists('./test_results/'):
    shutil.rmtree('./test_results/')
    print("test_results deleted")

Train3

In [ ]:
!python -X utf8 run.py --task_name classification --is_training 1 --model_id sleep_stage_test --model TimesNet --data sleep --root_path ./dataset/sleep_data_ready/ --seq_len 3840 --enc_in 8 --c_out 5 --batch_size 16 --d_model 32 --d_ff 64 --num_workers 0 --dropout 0.5 --learning_rate 0.00001 --train_epochs 20 --patience 5

In [ ]:
subdirs = [os.path.join(results_dir, d) for d in os.listdir(results_dir) if os.path.isdir(os.path.join(results_dir, d))]
subdirs.sort(key=os.path.getmtime)
latest_subdir = subdirs[-1]

pred_path = os.path.join(latest_subdir, 'pred.npy')
true_path = os.path.join(latest_subdir, 'true.npy')

y_pred = np.load(pred_path)
y_true = np.load(true_path)
if len(y_pred.shape) > 1 and y_pred.shape[1] > 1:
    y_pred = np.argmax(y_pred, axis=1)
labels = ['W', 'N1', 'N2', 'N3', 'R']

            
# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
            
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Sleep Stage Classification Confusion Matrix')
plt.xlabel('Predicted Stage (TimesNet)')
plt.ylabel('True Stage (Doctor)')
plt.show()

           
# Hypnogram
clinical_order = [0, 4, 1, 2, 3] # W, R, N1, N2, N3
y_true_clinical = [-clinical_order.index(val) for val in y_true]
y_pred_clinical = [-clinical_order.index(val) for val in y_pred]
clinical_labels = ['W', 'R', 'N1', 'N2', 'N3']

display_length = min(300, len(y_true))
y_true_display = y_true_clinical[:display_length]
y_pred_display = y_pred_clinical[:display_length]

plt.figure(figsize=(14, 5))
            
plt.step(range(display_length), y_true_display, label='True Stage (Doctor)', color='black', linewidth=2.5, where='post')

plt.step(range(display_length), [y - 0.1 for y in y_pred_display], label='Predicted (TimesNet)', color='red', linestyle='--', linewidth=2, where='post', alpha=0.8)

plt.yticks([-0, -1, -2, -3, -4], clinical_labels)
plt.title(f'Clinical Hypnogram over Time (First {display_length} Minutes of Test Set)')
plt.xlabel('Time (Minutes)')
plt.ylabel('Sleep Stage')
plt.legend(loc='lower left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()

Test with healthy set

In [ ]:
files_to_delete = [
    "X_test.npy", "y_test.npy"
]

print("Deleting...")
for file in files_to_delete:
    file_path = os.path.join(save_dir, file)
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f" - Cleaned: {file}")
    else:
        print(f" - Skiped: {file}")


np.save(os.path.join(save_dir, "X_test.npy"), X_test_healthy)
np.save(os.path.join(save_dir, "y_test.npy"), y_test_healthy_encoded)
print("ready for healthy test")

if os.path.exists('./test_results/'):
    shutil.rmtree('./test_results/')
    print("test_results deleted")

In [ ]:
!python -X utf8 run.py --task_name classification --is_training 0 --model_id sleep_stage_test --model TimesNet --data sleep --root_path ./dataset/sleep_data_ready/ --seq_len 3840 --enc_in 8 --c_out 5 --batch_size 16 --d_model 32 --d_ff 64 --num_workers 0

In [ ]:
subdirs = [os.path.join(results_dir, d) for d in os.listdir(results_dir) if os.path.isdir(os.path.join(results_dir, d))]
subdirs.sort(key=os.path.getmtime)
latest_subdir = subdirs[-1]

pred_path = os.path.join(latest_subdir, 'pred.npy')
true_path = os.path.join(latest_subdir, 'true.npy')

y_pred = np.load(pred_path)
y_true = np.load(true_path)
if len(y_pred.shape) > 1 and y_pred.shape[1] > 1:
    y_pred = np.argmax(y_pred, axis=1)
labels = ['W', 'N1', 'N2', 'N3', 'R']

            
# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
            
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Sleep Stage Classification Confusion Matrix')
plt.xlabel('Predicted Stage (TimesNet)')
plt.ylabel('True Stage (Doctor)')
plt.show()

           
# Hypnogram
clinical_order = [0, 4, 1, 2, 3] # W, R, N1, N2, N3
y_true_clinical = [-clinical_order.index(val) for val in y_true]
y_pred_clinical = [-clinical_order.index(val) for val in y_pred]
clinical_labels = ['W', 'R', 'N1', 'N2', 'N3']

display_length = min(300, len(y_true))
y_true_display = y_true_clinical[:display_length]
y_pred_display = y_pred_clinical[:display_length]

plt.figure(figsize=(14, 5))
            
plt.step(range(display_length), y_true_display, label='True Stage (Doctor)', color='black', linewidth=2.5, where='post')

plt.step(range(display_length), [y - 0.1 for y in y_pred_display], label='Predicted (TimesNet)', color='red', linestyle='--', linewidth=2, where='post', alpha=0.8)

plt.yticks([-0, -1, -2, -3, -4], clinical_labels)
plt.title(f'Clinical Hypnogram over Time (First {display_length} Minutes of Test Set)')
plt.xlabel('Time (Minutes)')
plt.ylabel('Sleep Stage')
plt.legend(loc='lower left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()

Test with severe set

In [ ]:
files_to_delete = [
    "X_test.npy", "y_test.npy"
]

print("Deleting...")
for file in files_to_delete:
    file_path = os.path.join(save_dir, file)
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f" - Cleaned: {file}")
    else:
        print(f" - Skiped: {file}")


np.save(os.path.join(save_dir, "X_test.npy"), X_test_severe)
np.save(os.path.join(save_dir, "y_test.npy"), y_test_severe_encoded)
print("ready for severe test")

if os.path.exists('./test_results/'):
    shutil.rmtree('./test_results/')
    print("test_results deleted")

In [ ]:
!python -X utf8 run.py --task_name classification --is_training 0 --model_id sleep_stage_test --model TimesNet --data sleep --root_path ./dataset/sleep_data_ready/ --seq_len 3840 --enc_in 8 --c_out 5 --batch_size 16 --d_model 32 --d_ff 64 --num_workers 0

In [ ]:
subdirs = [os.path.join(results_dir, d) for d in os.listdir(results_dir) if os.path.isdir(os.path.join(results_dir, d))]
subdirs.sort(key=os.path.getmtime)
latest_subdir = subdirs[-1]

pred_path = os.path.join(latest_subdir, 'pred.npy')
true_path = os.path.join(latest_subdir, 'true.npy')

y_pred = np.load(pred_path)
y_true = np.load(true_path)
if len(y_pred.shape) > 1 and y_pred.shape[1] > 1:
    y_pred = np.argmax(y_pred, axis=1)
labels = ['W', 'N1', 'N2', 'N3', 'R']

            
# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
            
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Sleep Stage Classification Confusion Matrix')
plt.xlabel('Predicted Stage (TimesNet)')
plt.ylabel('True Stage (Doctor)')
plt.show()

           
# Hypnogram
clinical_order = [0, 4, 1, 2, 3] # W, R, N1, N2, N3
y_true_clinical = [-clinical_order.index(val) for val in y_true]
y_pred_clinical = [-clinical_order.index(val) for val in y_pred]
clinical_labels = ['W', 'R', 'N1', 'N2', 'N3']

display_length = min(300, len(y_true))
y_true_display = y_true_clinical[:display_length]
y_pred_display = y_pred_clinical[:display_length]

plt.figure(figsize=(14, 5))
            
plt.step(range(display_length), y_true_display, label='True Stage (Doctor)', color='black', linewidth=2.5, where='post')

plt.step(range(display_length), [y - 0.1 for y in y_pred_display], label='Predicted (TimesNet)', color='red', linestyle='--', linewidth=2, where='post', alpha=0.8)

plt.yticks([-0, -1, -2, -3, -4], clinical_labels)
plt.title(f'Clinical Hypnogram over Time (First {display_length} Minutes of Test Set)')
plt.xlabel('Time (Minutes)')
plt.ylabel('Sleep Stage')
plt.legend(loc='lower left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()